
# Amazon ML Challenge 2026 — Business Entity Resolution (Colab + Google Drive edition)
### Retrieve-and-rerank pipeline: fine-tuned bge-m3 (blocking) + fine-tuned bge-reranker-v2-m3 (matching)

This is the Google Colab version of the pipeline — same logic as the Kaggle notebook, but storage is
built around a mounted Google Drive folder instead of `/kaggle/working` + Kaggle Datasets. Run cells
top to bottom. **Runtime → Change runtime type → T4 GPU** before you start (free tier).

**How this notebook is organized**
1. Mount Google Drive (manual click-through — see that cell)
2. Install packages
3. Config (all knobs live in one cell — `DEV_MODE` runs a fast subset first) + restore any
   already-synced progress from Drive
4. Get the raw data into place (Kaggle API pull, or manual upload — see that cell)
5. Load data + verify the country hard-filter assumption
6. Preprocessing / normalization (Unicode-safe — see note in that cell)
7. F_0.5 metric — implemented and self-tested against the problem statement's worked example
8. Train/validation split (held out from `train_source1`, used for every metric below)
9. Stage 1 — fine-tune `BAAI/bge-m3` as a domain-adapted bi-encoder
10. Stage 1 — encode the full S2+S3 corpus into a per-country FAISS IVF-PQ index (resumable, disk-safe)
11. Stage 1 — retrieve top-50 S2 + top-50 S3 candidates per S1 entity (raw, local scratch), then
    prefilter to the official, disk-cheap `candidate_pairs.tsv`
12. Recall@k check on the blocking stage (do this before spending GPU hours on the reranker)
13. Stage 2 — build reranker training pairs (real hard negatives mined from Stage 1, not random negatives)
14. Stage 2 — fine-tune `BAAI/bge-reranker-v2-m3` as the match/no-match cross-encoder
15. Stage 3 — score validation candidates, grid-search the decision threshold to maximize macro F_0.5
16. Final validation report + save predictions in the exact submission format
17. Compute budget notes + how to scale from `DEV_MODE` to the full dataset

**Every expensive cell is resumable**, same as the Kaggle version — it checks whether its output
already exists and skips recomputation. Colab's free tier caps sessions at ~12h with dynamic, unpublished
weekly GPU availability (heavier under load), and idle disconnects if the tab goes untouched for too
long, so you will very likely span more than one session for the full 2.2M-entity run.

**Storage is two tiers, same idea as the Kaggle version, different mechanics:**
- `LOCAL_DIR` (`/content/local`) — fast local VM disk (~60GB+ free on the free tier). ALL active work
  happens here: index building, model training, retrieval, everything. This disk is wiped when the
  Colab runtime disconnects.
- `DRIVE_DIR` (your mounted Drive folder) — the durable copy. A `sync_to_drive()` call after each
  expensive stage copies just the finished artifact (models, index, candidate files — not the
  regenerable scratch) there. At the start of every session, `restore_from_drive()` copies anything
  already on Drive back into `LOCAL_DIR` before the resumability checks run.

Writing straight to a Drive FUSE mount in a hot loop (like the chunked FAISS build) is slow and can be
flaky for large files — that's why all the heavy I/O stays local and Drive only sees complete,
finished artifacts at explicit sync points, not a stream of small incremental writes.



## 1. Mount Google Drive

**Manual step:** running this cell shows an in-notebook permission prompt — click **"Connect to Google
Drive"** / **Allow**, and complete the Google sign-in if asked. Use the *same* Google account for Colab
and Drive; mounting a different account than the one you're signed into Colab with is blocked (a
security fix Google shipped a while back — the old "mount someone else's Drive" workaround no longer
works). If you hit `MessageError: credential propagation was unsuccessful`, retry with
`force_remount=True`, try an incognito window, and check Chrome isn't blocking third-party cookies for
`colab.research.google.com` — that error has been reported intermittently across browser updates.


In [ ]:

# ---- Cell 1: mount Google Drive ----
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted. Your files are under /content/drive/MyDrive/")


In [ ]:

# ---- Cell 2: install packages -------------------------------------------------
# Colab already ships torch + CUDA, but pin sentence-transformers >= 6.0 explicitly: the modern
# Trainer-based fine-tuning API used below (SentenceTransformerTrainer / CrossEncoderTrainer)
# doesn't exist in the older 2.x/3.x version Colab's base image may have.
!pip install -q -U "sentence-transformers>=6.0" "transformers>=4.45" datasets faiss-gpu-cu12 lightgbm 2>/dev/null \
    || pip install -q -U "sentence-transformers>=6.0" "transformers>=4.45" datasets faiss-cpu lightgbm
print("done")


In [ ]:

# ---- Cell 3: config -------------------------------------------------------
import os, re, json, math, random, unicodedata, time, shutil
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- scale knobs (defined before paths below, since DEV vs FULL artifacts must not collide) ----
# DEV_MODE runs the ENTIRE pipeline end-to-end on a small slice of S1 AND a subsampled S2/S3 corpus
# (every true match for the sampled S1 entities is guaranteed kept, plus random filler per country for
# realistic retrieval difficulty — see Cell 10) so a full correctness + F_0.5 pass, INCLUDING the index
# build (normally the single biggest cost), takes minutes, not hours. Flip to False for the full run.
DEV_MODE = True
N_S1_DEV = 20_000          # number of S1 entities to use when DEV_MODE=True
DEV_CORPUS_FILLER_PER_COUNTRY = 20_000   # random non-match filler records per country, DEV_MODE only
VAL_FRACTION = 0.05        # held-out fraction of S1 entities for evaluation (both modes)
MODE_TAG = "dev" if DEV_MODE else "full"   # kept OUT of paths that mean the same thing either way
SKIP_BIENCODER_FINETUNE = False   # True -> use off-the-shelf bge-m3 for retrieval, no fine-tuning at all.
                    # Recommended workflow: a full DEV_MODE pass is cheap enough to run TWICE — once
                    # True, once False — comparing recall@k (Cell 12) each time. If off-the-shelf already
                    # clears ~97%+, fine-tuning the bi-encoder buys little to nothing on the final score
                    # (it's the reranker, not the retriever, that precision actually comes from) and you
                    # can skip that GPU time on the full run entirely.
# Everything downstream of the retriever (index, candidates, reranker data + models) depends on WHICH
# bi-encoder built it, so off-the-shelf and fine-tuned runs get separate folders — otherwise flipping
# SKIP_BIENCODER_FINETUNE would silently reuse the other run's stale index/candidates.
RUN_TAG = MODE_TAG + ("_offshelf" if SKIP_BIENCODER_FINETUNE else "")

# ---- paths ----
# Rename this if you want a different folder name in your Drive — everything else derives from it.
DRIVE_DIR = "/content/drive/MyDrive/amazon_ml_challenge"
LOCAL_DIR = "/content/local"    # fast local VM disk — ALL active work happens here, wiped on disconnect
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

# redirect the HF model-download cache to local disk — re-downloading a base checkpoint costs
# bandwidth/time, not Drive space, and once you have your OWN fine-tuned checkpoint saved you
# don't need the base-model cache at all.
os.environ["HF_HOME"] = f"{LOCAL_DIR}/hf_cache"

# reduce PyTorch CUDA memory fragmentation on long training runs — set before any CUDA context exists
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- local working dirs (all under LOCAL_DIR — fast, ephemeral) ----
# split.json and the preprocessed text are the same underlying full-dataset artifacts either way (DEV_MODE
# just samples from within them at use-time), so they're NOT mode-tagged. Everything else is built FROM
# whichever S1 entities/corpus are actually in play this run, so DEV and FULL genuinely differ and must
# not collide — a stale small DEV_MODE index/model/candidate-file silently reused for a full run (or
# vice versa) would be a correctness bug, not just a wasted skip.
RAW_DATA_ON_DRIVE = False  # True if you manually uploaded the 4 TSVs to Drive yourself instead of using
                            # the Kaggle-API cell below — see that cell's markdown for the exact folder.
INPUT_DIR = f"{DRIVE_DIR}/raw_data" if RAW_DATA_ON_DRIVE else f"{LOCAL_DIR}/raw_data"
SPLIT_DIR = f"{LOCAL_DIR}/split"
PREP_DIR = f"{LOCAL_DIR}/preprocessed"
INDEX_DIR = f"{LOCAL_DIR}/index_{RUN_TAG}"
MODEL_DIR = f"{LOCAL_DIR}/models_{RUN_TAG}"
CAND_DIR = f"{LOCAL_DIR}/candidates_{RUN_TAG}"
RAW_CAND_DIR = f"{LOCAL_DIR}/raw_candidates_{RUN_TAG}"
RERANK_DIR = f"{LOCAL_DIR}/reranker_data_{RUN_TAG}"
for d in (INPUT_DIR, PREP_DIR, SPLIT_DIR, INDEX_DIR, CAND_DIR, RAW_CAND_DIR, RERANK_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

# ---- which of the above are worth persisting to Drive (expensive to redo), vs. pure scratch ----
# (matched by folder basename: LOCAL_DIR/<name> <-> DRIVE_DIR/<name>) — mode-tagged names flow through
# automatically, so a DEV run and a FULL run sync to separate Drive folders too, same reasoning as above.
TEST_INDEX_DIR = f"{LOCAL_DIR}/test_index_{RUN_TAG}"   # test-corpus index (Section 19)
PERSIST_SUBDIRS = ["split", "preprocessed", f"index_{RUN_TAG}", f"models_{RUN_TAG}", f"candidates_{RUN_TAG}",
                   f"test_index_{RUN_TAG}"]

def restore_from_drive():
    # Run once per session, right after mount: pulls anything already synced back onto local disk
    # so the resumability checks throughout this notebook (`if os.path.exists(...)`) see prior progress.
    # Local dirs are freshly created (empty) each session by the os.makedirs loop above, so "local dir
    # exists but empty" reliably means "nothing restored yet this session" — safe to overwrite.
    for sub in PERSIST_SUBDIRS:
        drive_path, local_path = f"{DRIVE_DIR}/{sub}", f"{LOCAL_DIR}/{sub}"
        if os.path.isdir(drive_path) and not (os.path.isdir(local_path) and os.listdir(local_path)):
            shutil.copytree(drive_path, local_path, dirs_exist_ok=True)
            print(f"Restored {sub}/ from Drive.")

def sync_to_drive(local_dir):
    # Call after a stage finishes writing to a LOCAL_DIR subfolder to make it durable across sessions.
    # Takes the actual directory (e.g. MODEL_DIR), not a hand-typed name — the folders are run-tagged
    # (models_dev, index_full_offshelf, ...), and a hand-typed "models" would silently sync nothing.
    sub = os.path.basename(os.path.normpath(local_dir))
    local_path, drive_path = f"{LOCAL_DIR}/{sub}", f"{DRIVE_DIR}/{sub}"
    if not os.path.isdir(local_path):
        raise FileNotFoundError(f"sync_to_drive: {local_path} does not exist")
    shutil.copytree(local_path, drive_path, dirs_exist_ok=True)
    print(f"Synced {sub}/ to Drive.")

restore_from_drive()

# ---- retrieval / blocking ----
TOP_K_PER_SOURCE = 50      # candidates retrieved per source (S2, S3) per S1 entity, as requested
IVFPQ_M = 64                # PQ sub-quantizers (64 -> ~64 bytes/vector instead of 4096 raw float32)
IVFPQ_NBITS = 8
TRAIN_SAMPLE_FOR_QUANTIZER = 200_000   # vectors used to train each country's IVF-PQ quantizer
ENCODE_BATCH_SIZE = 256
ADD_CHUNK_SIZE = 50_000     # encode+add to the index in chunks so we never hold the full corpus in RAM

# ---- models ----
BIENCODER_BASE = "BAAI/bge-m3"                 # 568M params, MIT license, 100+ languages
RERANKER_BASE = "BAAI/bge-reranker-v2-m3"      # 568M params, permissive license, cross-encoder
# Heavier optional swap (see the last markdown cell for the tradeoff before you flip this):
# RERANKER_BASE = "Qwen/Qwen3-Reranker-4B"
SAVE_FP16 = True   # halve each saved checkpoint's disk footprint (~2.2GB -> ~1.1GB); no meaningful
                    # quality loss for embedding/reranking inference, and training still runs in fp16
                    # mixed precision either way (see `fp16=True` in the TrainingArguments below)
# (SKIP_BIENCODER_FINETUNE lives at the top of this cell — it decides the run-tagged folder names)
RERANKER_APPROACH = "gbdt"   # "gbdt" or "cross_encoder" — see Cell 14's markdown for the tradeoff.
                    # Recommended default: "gbdt". No GPU-memory risk, trains in seconds, and is a
                    # completely standard, competitive choice for this kind of record-linkage problem —
                    # not a fallback. Cells 14a/15 (cross-encoder) and 14b/15b (GBDT) both self-skip
                    # based on this flag, so you can leave every cell in and just flip this one setting.

# ---- reranker inference funnel ----
RERANK_PREFILTER_K = 15    # cut each entity's candidates down to this many (by bi-encoder score)
                            # BEFORE running the heavier cross-encoder over them

# ---- training hyperparams ----
# MAX_SEQ_LENGTH matters more than it looks: without an explicit cap, bge-m3's 8192-token default
# context means one unusually long address in a batch pads every other record in that SAME batch out to
# match it, spiking that batch's memory far above a "typical" batch — the classic signature of a CUDA
# OOM that survives many fine training steps before failing on one particular batch (not a sign the
# batch size itself is wrong). Per the dataset audit, real name+address text needs nowhere near this:
# ~3.5 name words / ~8 address words on average, ~46 words even at the observed max.
MAX_SEQ_LENGTH = 128        # bi-encoder cap, in tokens; the cross-encoder uses 2x this (concatenated pair)
BIENCODER_EPOCHS = 1
BIENCODER_BATCH_SIZE = 128  # safe on a 16GB T4 even before the MAX_SEQ_LENGTH cap above. On a 22GB L4,
                             # 160-192 is reasonable WITH that cap in place — don't raise batch size on
                             # its own without it, that's what caused the OOM in the first place.
BIENCODER_LR = 2e-5
RERANKER_EPOCHS = 1
RERANKER_BATCH_SIZE = 64    # same logic: fine to raise toward ~96 on an L4, but only alongside the cap
RERANKER_LR = 2e-5
HARD_NEGATIVES_PER_POSITIVE = 8   # cap on negatives sampled per positive for reranker training

print("Config loaded. DEV_MODE =", DEV_MODE, "| RUN_TAG =", RUN_TAG)
print(f"Local working dir (fast, ephemeral)  : {LOCAL_DIR}")
print(f"Drive-backed dir  (durable, synced)  : {DRIVE_DIR}")
# Rough full-scale budget for what sync_to_drive() actually copies to Drive with this config:
#   Preprocessed text (preprocessed/)               ~2-3 GB   (full ~12.5M-row dataset, same either mode)
#   FAISS index (index_full/)                       ~1.2 GB   (fixed, independent of S1 count)
#   2x fine-tuned models, fp16 (models_full/)       ~2.2 GB   (568M params each)
#   candidate_pairs.tsv + scores (candidates_full/) ~2-3 GB   (train+val, at RERANK_PREFILTER_K=15/entity)
#   split.json                                       <0.1 GB
#   ------------------------------------------------------
#   total                                            ~7.5-8.5 GB  — check this fits your Drive's free space
#                                                     (free Google accounts get 15GB shared with Gmail/Photos)



## 4. Get the raw data into place

Two ways to get the 4 source TSVs (~1.3GB total) where the notebook expects them. Pick one.

**Option A — Kaggle API (default, `RAW_DATA_ON_DRIVE = False`):** pulls straight from your Kaggle
dataset into local disk, fresh each session — cheap to redo, no manual upload, but not persisted (by
design: keeping a permanent copy on Drive would burn Drive quota you don't need to spend). Needs a
Kaggle API token: **kaggle.com → your profile → Settings → API → Create New Token**, which downloads
`kaggle.json`. Manual step: add `KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab **Secrets** (🔑 icon, left
sidebar — both values are inside `kaggle.json`), toggle "Notebook access" on for each, then run the
cell below.

**Option B — manual upload to Drive (`RAW_DATA_ON_DRIVE = True` in the config cell above):** upload the
same 4 files you already have in your Kaggle dataset — `train_source1.tsv`, `train_source2.tsv`,
`train_source3.tsv`, `train_ground_truth.tsv` — into this exact folder in your Google Drive:
`MyDrive/amazon_ml_challenge/raw_data/`. Two ways to actually do that upload:
- Open **drive.google.com** in another browser tab (same Google account), create the `amazon_ml_challenge`
  folder if it isn't there yet from mounting, then the `raw_data` subfolder inside it, and drag the 4
  files in. This is the more reliable route for files this size — the browser's own upload handles
  large files better than Colab's in-notebook widget.
- Or, in Colab's **Files** pane (folder icon, left sidebar) — once Drive is mounted you can browse into
  `drive/MyDrive/`, navigate to (or create) that same folder, right-click it, and upload from there.

Once they're in Drive, set `RAW_DATA_ON_DRIVE = True` in the config cell and rerun it — `INPUT_DIR` then
points at the Drive copy instead of local disk, so it persists across sessions and you never re-upload
again. Cell 4 below checks whether the files are already present at `INPUT_DIR` either way and only
downloads via Option A if they're missing, so with `RAW_DATA_ON_DRIVE = True` it just confirms they're
there and skips the download entirely.


In [ ]:

# ---- Cell 4: pull the raw dataset (Kaggle API — Option A; no-op if Option B's files are already there) ----
KAGGLE_DATASET_SLUG = "dhruvgupta1606/amazon-train-1"   # <-- adjust if yours differs

if os.path.exists(f"{INPUT_DIR}/train_source1.tsv"):
    print(f"Raw data already present at {INPUT_DIR}, skipping download.")
else:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    !pip install -q kaggle
    !kaggle datasets download -d {KAGGLE_DATASET_SLUG} -p {INPUT_DIR} --unzip
    # the zip usually expands into an extra 'train/' subfolder — flatten it if so
    nested = f"{INPUT_DIR}/train"
    if os.path.isdir(nested):
        for fn in os.listdir(nested):
            shutil.move(f"{nested}/{fn}", f"{INPUT_DIR}/{fn}")
        os.rmdir(nested)
    print("Downloaded to", INPUT_DIR)

print(sorted(os.listdir(INPUT_DIR)))



## 5. Load data + verify the country hard-filter assumption

Before anything else: the plan is to block candidates by requiring `S1.country == candidate.country`.
That's a huge, safe-looking recall boost — **unless** some ground-truth matches actually cross
countries, in which case a hard filter silently zeroes out recall on exactly those entities. Check
this on the real data before you bake the assumption into the rest of the pipeline.


In [ ]:

# ---- Cell 5: load data ----
s1 = pd.read_csv(f"{INPUT_DIR}/train_source1.tsv", sep="\t", dtype=str)
s2 = pd.read_csv(f"{INPUT_DIR}/train_source2.tsv", sep="\t", dtype=str)
s3 = pd.read_csv(f"{INPUT_DIR}/train_source3.tsv", sep="\t", dtype=str)
gt = pd.read_csv(f"{INPUT_DIR}/train_ground_truth.tsv", sep="\t", dtype=str).fillna("")

for name, df in (("S1", s1), ("S2", s2), ("S3", s3), ("ground truth", gt)):
    print(f"{name}: {len(df):,} rows, columns={list(df.columns)}")

assert set(s1.columns) >= {"entity_id", "business_name", "business_address", "country"}
assert set(gt.columns) == {"source1_entity_id", "matched_entity_ids"}
n_singleton = (gt["matched_entity_ids"] == "").sum()
print(f"\nSingletons in ground truth: {n_singleton:,} / {len(gt):,} ({n_singleton/len(gt):.1%})")


In [ ]:

# ---- Cell 6: verify the country hard-filter assumption on REAL ground truth ----
id2country = pd.concat([
    s1[["entity_id", "country"]], s2[["entity_id", "country"]], s3[["entity_id", "country"]]
]).set_index("entity_id")["country"].to_dict()

cross_country_violations = 0
checked = 0
for _, row in gt.iterrows():
    s1_country = id2country.get(row["source1_entity_id"])
    matched = [m for m in row["matched_entity_ids"].split(",") if m]
    for m in matched:
        checked += 1
        if id2country.get(m) != s1_country:
            cross_country_violations += 1

print(f"Checked {checked:,} ground-truth match edges.")
print(f"Cross-country violations: {cross_country_violations}")
if cross_country_violations == 0:
    print("-> Hard country filter is SAFE: no ground-truth match ever crosses country. Proceeding with it.")
    COUNTRY_HARD_FILTER = True
else:
    pct = cross_country_violations / max(checked, 1)
    print(f"-> {pct:.4%} of matches cross country. A hard filter would cap your recall by this much.")
    print("   Decide deliberately: keep the hard filter and accept the recall ceiling, or soften it")
    print("   (e.g. retrieve globally and use country-match as a FEATURE instead of a filter).")
    COUNTRY_HARD_FILTER = True  # <-- change this yourself if the check above found violations



## 6. Preprocessing / normalization

**Unicode note:** the dataset audit's token-frequency table shows single bare Devanagari letters
(ट, र, ल, ड, म...) as extremely high-frequency "tokens." That's a tell that whatever tokenizer produced
that table was breaking Devanagari conjuncts apart — combining marks (matras, virama) getting stripped
by a naive `\w`/punctuation regex, which doesn't reliably treat Indic combining marks as word
characters. The normalizer below strips punctuation by explicit Unicode category (keep letters L*,
marks M*, numbers N*) instead of relying on `\w`, specifically so it doesn't repeat that bug and
mangle Hindi-script business names/addresses.


In [ ]:

# ---- Cell 5: normalization ----
SUFFIX_MAP = {
    "pvt": "private", "private": "private",
    "ltd": "limited", "limited": "limited",
    "llc": "llc",
    "inc": "incorporated", "incorporated": "incorporated",
    "corp": "corporation", "corporation": "corporation",
    "co": "company", "company": "company",
    "llp": "llp", "dba": "dba",
}
ADDR_ABBR_MAP = {
    "rd": "road", "st": "street", "ave": "avenue", "blvd": "boulevard",
    "dr": "drive", "ln": "lane", "apt": "apartment", "ste": "suite",
    "hwy": "highway", "ct": "court", "pl": "place", "sq": "square",
}
_ws_re = re.compile(r"\s+")
_digit_re = re.compile(r"\d+")

def _strip_punct_unicode_safe(s: str) -> str:
    # keep letters/marks/numbers/whitespace; drop everything else. Using unicodedata.category()
    # instead of the `\w` regex class specifically preserves Indic combining marks (see note above).
    return "".join(ch if (unicodedata.category(ch)[0] in ("L", "M", "N") or ch.isspace()) else " " for ch in s)

def normalize_text(s) -> str:
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFKC", str(s)).lower().replace("&", " and ")
    s = _strip_punct_unicode_safe(s)
    return _ws_re.sub(" ", s).strip()

def normalize_name(name) -> str:
    s = normalize_text(name)
    return " ".join(SUFFIX_MAP.get(tok, tok) for tok in s.split())

def normalize_address(addr) -> str:
    s = normalize_text(addr)
    return " ".join(ADDR_ABBR_MAP.get(tok, tok) for tok in s.split())

def extract_digits(s) -> set:
    # house numbers / PIN / ZIP codes as an exact-match signal for later feature engineering
    if pd.isna(s):
        return set()
    return set(_digit_re.findall(str(s)))

def preprocess_df(df):
    df = df.copy()
    df["name_norm"] = df["business_name"].map(normalize_name)
    df["addr_norm"] = df["business_address"].map(normalize_address)
    df["digits"] = df["business_address"].map(lambda x: ",".join(sorted(extract_digits(x))))
    # Explicit field labels, not a bare concatenation — gives both the bi-encoder and the cross-encoder
    # a structural cue that "name" and "address" are two different kinds of signal (identity vs.
    # location), rather than asking the model to infer that purely from position/statistics. Cheap to
    # add, no downside, and this is standard practice for embedding structured records.
    df["text"] = ("name: " + df["name_norm"] + " | address: " + df["addr_norm"]).str.strip()
    return df

_prep_path = lambda name: f"{PREP_DIR}/{name}_prep.parquet"
if all(os.path.exists(_prep_path(n)) for n in ("s1", "s2", "s3")):
    print("Preprocessed files already exist, loading from disk...")
    s1p = pd.read_parquet(_prep_path("s1"))
    s2p = pd.read_parquet(_prep_path("s2"))
    s3p = pd.read_parquet(_prep_path("s3"))
else:
    s1p, s2p, s3p = preprocess_df(s1), preprocess_df(s2), preprocess_df(s3)
    s1p.to_parquet(_prep_path("s1")); s2p.to_parquet(_prep_path("s2")); s3p.to_parquet(_prep_path("s3"))
    print("Preprocessed and saved to", PREP_DIR)
    sync_to_drive(PREP_DIR)

print(s1p[["entity_id", "business_name", "name_norm", "addr_norm"]].head(3).to_string())



## 7. The F_0.5 metric — implemented exactly as specified, self-tested

Macro-average of a per-entity F_0.5, singletons included. Predicting an empty set for a true
singleton scores 1.0; predicting *anything* for a singleton scores 0.0; missing every true match on a
non-singleton entity also scores 0.0 (same penalty as being fully wrong) — so "predict nothing when
unsure" is not a free lunch, it only works for the ~5-6% of entities that are actually singletons.


In [ ]:

# ---- Cell 6: F_0.5 metric + self-test against the problem statement's worked example ----
def f_beta_score(true_set: set, pred_set: set, beta: float = 0.5) -> float:
    if len(true_set) == 0:
        return 1.0 if len(pred_set) == 0 else 0.0
    if len(pred_set) == 0:
        return 0.0
    tp = len(true_set & pred_set)
    precision = tp / len(pred_set)
    recall = tp / len(true_set)
    b2 = beta * beta
    denom = b2 * precision + recall
    return (1 + b2) * precision * recall / denom if denom > 0 else 0.0

def macro_f_beta(rows, beta: float = 0.5) -> float:
    return sum(f_beta_score(t, p, beta) for t, p in rows) / len(rows)

# self-test: worked example from the problem statement
_true = {"S2-00047", "S3-00812"}
_pred = {"S2-00047", "S2-00193", "S3-00812"}
_score = f_beta_score(_true, _pred)
assert abs(_score - 0.7142857142857143) < 1e-9, f"F0.5 implementation mismatch: got {_score}"
assert f_beta_score(set(), set()) == 1.0
assert f_beta_score(set(), {"X"}) == 0.0
assert f_beta_score({"X"}, set()) == 0.0
print(f"Worked-example check passed: F_0.5 = {_score:.4f} (spec says 0.714)")
print("F_0.5 metric verified.")



## 8. Train / validation split

Split at the **S1-entity level**, not the record level: the S2/S3 corpus stays whole (it's the fixed
"database" both splits retrieve against, mirroring how the real test corpus works), you're only
holding out *which S1 queries* you evaluate on.


In [ ]:

# ---- Cell 7: train/val split ----
split_path = f"{SPLIT_DIR}/split.json"
if os.path.exists(split_path):
    split = json.load(open(split_path))
    train_ids, val_ids = split["train_ids"], split["val_ids"]
else:
    all_s1_ids = s1p["entity_id"].tolist()
    rng = random.Random(SEED)
    rng.shuffle(all_s1_ids)
    n_val = max(1, int(len(all_s1_ids) * VAL_FRACTION))
    val_ids = all_s1_ids[:n_val]
    train_ids = all_s1_ids[n_val:]
    json.dump({"train_ids": train_ids, "val_ids": val_ids}, open(split_path, "w"))
    sync_to_drive(SPLIT_DIR)

if DEV_MODE:
    rng = random.Random(SEED)
    train_ids_use = rng.sample(train_ids, min(N_S1_DEV, len(train_ids)))
    val_ids_use = rng.sample(val_ids, min(max(1, N_S1_DEV // 10), len(val_ids)))
else:
    train_ids_use, val_ids_use = train_ids, val_ids

print(f"Full split: {len(train_ids):,} train / {len(val_ids):,} val S1 entities")
print(f"Using this run: {len(train_ids_use):,} train / {len(val_ids_use):,} val (DEV_MODE={DEV_MODE})")

gt_map = {}
for _, row in gt.iterrows():
    gt_map[row["source1_entity_id"]] = set(x for x in row["matched_entity_ids"].split(",") if x)



## 9. Stage 1 — fine-tune `bge-m3` as a domain-adapted bi-encoder

Off-the-shelf embeddings won't reliably separate "ABC Enterprises Pvt Ltd" from "ABC Enterprises LLP" —
that's exactly the kind of confusion a general-purpose sentence embedding wasn't trained to make. A
short contrastive fine-tune on your own ground-truth positive pairs, using in-batch negatives
(`MultipleNegativesRankingLoss`), is generally worth it for your blocking recall ceiling — but it's not
free: it's real GPU time, and blocking only needs to be *good enough to not filter out true matches*,
not perfectly discriminating (that precision burden falls on the reranker in Stage 2, where fine-tuning
time is much better spent if you're choosing where to spend a limited budget).

**Don't take fine-tuning on faith — measure it.** `DEV_MODE` now subsamples the S2/S3 corpus too (see
Cell 10), not just the S1 side, so a full pass including the index build is minutes, not hours. Run it
once with `SKIP_BIENCODER_FINETUNE = True` and once `False`, compare the recall@k each gives you in
Cell 12: if off-the-shelf bge-m3 already clears ~97%+, fine-tuning it buys little to nothing on your
final score at full scale, and you can skip straight to the reranker with the GPU-hours you just saved.

Whichever way you decide, note this step — when it does run — does **not** need hard negatives:
in-batch negatives are the standard way to contrastively fine-tune a bi-encoder and don't require an
existing retriever (no chicken-and-egg problem). Hard negatives matter for the *reranker* in Stage 2,
once a retriever exists to mine them with.


In [ ]:

# ---- Cell 8: build bi-encoder fine-tuning data (positive pairs only, in-batch negatives) ----
from datasets import Dataset

id2text = {}
for df in (s1p, s2p, s3p):
    id2text.update(dict(zip(df["entity_id"], df["text"])))

biencoder_pairs_path = f"{PREP_DIR}/biencoder_pairs_{MODE_TAG}.parquet"
if os.path.exists(biencoder_pairs_path):
    pairs_df = pd.read_parquet(biencoder_pairs_path)
else:
    anchors, positives = [], []
    for s1_id in train_ids_use:   # mode-aware: the small DEV_MODE slice, or the full train split
        for m in gt_map.get(s1_id, set()):
            if s1_id in id2text and m in id2text:
                anchors.append(id2text[s1_id])
                positives.append(id2text[m])
    pairs_df = pd.DataFrame({"anchor": anchors, "positive": positives})
    pairs_df.to_parquet(biencoder_pairs_path)

print(f"Bi-encoder training pairs: {len(pairs_df):,}")
biencoder_train_ds = Dataset.from_pandas(pairs_df[["anchor", "positive"]], preserve_index=False)


In [ ]:

# ---- Cell 9: fine-tune bge-m3 ----
from sentence_transformers import (
    SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments,
)
from sentence_transformers.sentence_transformer import losses as st_losses

FINETUNED_BIENCODER_PATH = f"{MODEL_DIR}/bge-m3-finetuned"

if SKIP_BIENCODER_FINETUNE:
    print("SKIP_BIENCODER_FINETUNE=True — using off-the-shelf bge-m3 for retrieval, no training done.")
    bi_model = SentenceTransformer(BIENCODER_BASE)
    bi_model.max_seq_length = MAX_SEQ_LENGTH
elif os.path.exists(FINETUNED_BIENCODER_PATH):
    print("Fine-tuned bi-encoder already exists, loading it.")
    bi_model = SentenceTransformer(FINETUNED_BIENCODER_PATH)
    bi_model.max_seq_length = MAX_SEQ_LENGTH
else:
    bi_model = SentenceTransformer(BIENCODER_BASE)
    # Cap sequence length explicitly. bge-m3 defaults to an 8192-token context window — nothing this
    # pipeline ever encodes needs more than a small fraction of that (name+address text, per the dataset
    # audit: ~3.5 name words, ~8 address words on average, ~46 words even at the observed max). Left
    # uncapped, one unusually long record in a batch pads every OTHER record in that batch out to match
    # it, spiking that batch's activation memory far above a "typical" batch — which is exactly what an
    # OOM that survives dozens of fine steps before failing on one particular batch usually means, rather
    # than the batch size itself being wrong. This bounds the worst case, not just the average case.
    bi_model.max_seq_length = MAX_SEQ_LENGTH
    loss = st_losses.MultipleNegativesRankingLoss(bi_model)

    args = SentenceTransformerTrainingArguments(
        output_dir=f"{LOCAL_DIR}/bge-m3-ckpts",   # trainer logs/state — local scratch, not the final weights
        num_train_epochs=BIENCODER_EPOCHS,
        per_device_train_batch_size=BIENCODER_BATCH_SIZE,
        learning_rate=BIENCODER_LR,
        warmup_ratio=0.05,
        fp16=True,
        save_strategy="no",
        logging_steps=50,
        report_to=[],
    )
    trainer = SentenceTransformerTrainer(
        model=bi_model, args=args, train_dataset=biencoder_train_ds, loss=loss,
    )
    trainer.train()
    if SAVE_FP16:
        bi_model = bi_model.half()   # halves the checkpoint on disk; fine for inference-only use from here
    bi_model.save(FINETUNED_BIENCODER_PATH)
    print("Saved fine-tuned bi-encoder to", FINETUNED_BIENCODER_PATH)
    sync_to_drive(MODEL_DIR)



## 10. Stage 1 — encode the corpus + build a per-country FAISS index

**Why IVF-PQ and not a flat/raw index:** the full S2+S3 corpus is ~10.3M records. Raw float32 vectors
at bge-m3's 1024 dims would need **~42 GB** — more than Kaggle's persistent working-directory budget.
An IVF-PQ index (compressed, `m=64` sub-quantizers at `nbits=8`) needs roughly **~1.2 GB** for the same
corpus (measured ~114 bytes/vector vs. 4096 raw) at a small, usually-worthwhile recall cost versus an
exact index. The quantizer is trained on a random sample, then vectors are encoded and added in chunks
so the full corpus is never held in memory at once.

This cell is resumable per-country: if `{country}.faiss` already exists it's loaded straight from disk.
Under `DEV_MODE` the corpus itself is subsampled first (see the cell below) — that index and its
resumability are scoped to the smaller corpus, so switching `DEV_MODE` off later rebuilds against the
real full corpus rather than silently reusing the small one.


In [ ]:

# ---- Cell 10: encode corpus + build per-country FAISS IVF-PQ index (chunked, resumable) ----
import faiss

FLAT_INDEX_MAX_N = 20_000   # below this, an exact flat index is small + fast, and IVF-PQ can't train
                            # reliably anyway (PQ needs >=256 points per codebook, IVF ~39 per list)

def _encode(texts, model):
    return model.encode(
        texts, batch_size=ENCODE_BATCH_SIZE, normalize_embeddings=True,
        show_progress_bar=False, convert_to_numpy=True,
    ).astype("float32")

def build_country_index(country, df_c, model, id_col="entity_id", text_col="text"):
    n = len(df_c)
    if n == 0:
        return None, np.array([])
    d = model.get_sentence_embedding_dimension()
    if n <= FLAT_INDEX_MAX_N:
        index = faiss.IndexFlatIP(d)
        for start in range(0, n, ADD_CHUNK_SIZE):
            index.add(_encode(df_c[text_col].values[start:start + ADD_CHUNK_SIZE].tolist(), model))
        return index, df_c[id_col].values
    nlist = max(4, min(4096, int(4 * math.sqrt(n))))
    quantizer = faiss.IndexFlatIP(d)
    index = faiss.IndexIVFPQ(quantizer, d, nlist, IVFPQ_M, IVFPQ_NBITS, faiss.METRIC_INNER_PRODUCT)

    train_n = min(TRAIN_SAMPLE_FOR_QUANTIZER, n)
    train_idx = np.random.choice(n, size=train_n, replace=False)
    train_vecs = _encode(df_c[text_col].values[train_idx].tolist(), model)
    index.train(train_vecs)
    del train_vecs

    for start in range(0, n, ADD_CHUNK_SIZE):
        chunk = df_c[text_col].values[start:start + ADD_CHUNK_SIZE].tolist()
        vecs = _encode(chunk, model)
        index.add(vecs)
        del vecs
        if start % (ADD_CHUNK_SIZE * 5) == 0:
            print(f"  [{country}] indexed {start + len(chunk):,}/{n:,}")

    index.nprobe = min(64, nlist)
    return index, df_c[id_col].values

def build_or_load_indexes(corpus, countries, index_dir, model):
    # Per-country, resumable: each finished country is written to index_dir immediately, so a
    # disconnect mid-build only loses the country in progress.
    indexes = {}   # country -> (faiss_index, ids_array, source_array)
    for country in sorted(countries):
        idx_path = f"{index_dir}/{country}.faiss"
        ids_path = f"{index_dir}/{country}_ids.npy"
        src_path = f"{index_dir}/{country}_src.npy"
        if os.path.exists(idx_path) and os.path.exists(ids_path) and os.path.exists(src_path):
            print(f"[{country}] index already built, loading from disk.")
            idx = faiss.read_index(idx_path)
            if isinstance(idx, faiss.IndexIVF):
                idx.nprobe = min(64, idx.nlist)
            ids = np.load(ids_path, allow_pickle=True)
            srcs = np.load(src_path, allow_pickle=True)
        else:
            df_c = corpus[corpus["country"] == country].reset_index(drop=True)
            print(f"[{country}] building index over {len(df_c):,} records...")
            t0 = time.time()
            idx, ids = build_country_index(country, df_c, model)
            srcs = df_c["source"].values
            if idx is not None:
                faiss.write_index(idx, idx_path)
                np.save(ids_path, ids)
                np.save(src_path, srcs)
            print(f"[{country}] done in {time.time()-t0:.1f}s")
        indexes[country] = (idx, ids, srcs)
    for c, (idx, ids, srcs) in indexes.items():
        p = f"{index_dir}/{c}.faiss"
        sz = os.path.getsize(p) / 1e6 if os.path.exists(p) else 0
        print(f"{c}: {len(ids):,} vectors indexed, {sz:.1f} MB on disk")
    return indexes

# combine S2+S3 into one taggable corpus so each country index covers both sources at once
corpus = pd.concat([
    s2p.assign(source="S2"), s3p.assign(source="S3"),
], ignore_index=True)

if DEV_MODE:
    # Subsample the corpus too, not just S1 — otherwise DEV_MODE still pays the full index-build cost
    # (the single biggest stage per the compute-budget section) even though only 20k S1 queries run
    # against it. Every true match for the S1 entities in play this run is guaranteed kept; the rest
    # is random filler per country so retrieval is still realistically hard, not a trivial lookup.
    needed_ids = set()
    for eid in train_ids_use + val_ids_use:
        needed_ids |= gt_map.get(eid, set())
    rng = random.Random(SEED)
    keep_ids = set(needed_ids)
    for country in s1p["country"].unique():
        pool = corpus.loc[
            (corpus["country"] == country) & (~corpus["entity_id"].isin(keep_ids)), "entity_id"
        ].tolist()
        keep_ids |= set(rng.sample(pool, min(DEV_CORPUS_FILLER_PER_COUNTRY, len(pool))))
    corpus = corpus[corpus["entity_id"].isin(keep_ids)].reset_index(drop=True)
    print(f"DEV_MODE: corpus subsampled to {len(corpus):,} records "
          f"(from {len(s2p) + len(s3p):,}) — every true match for this run's S1 entities is kept.")

country_indexes = build_or_load_indexes(corpus, s1p["country"].unique(), INDEX_DIR, bi_model)
sync_to_drive(INDEX_DIR)



## 11a. Retrieve top-50 S2 + top-50 S3 candidates per S1 entity (raw, to scratch)

The raw top-`TOP_K_PER_SOURCE` retrieval per source is large (2.2M entities × 100 candidates at full
scale) and fully regenerable from the persisted index + model, so it's written to `RAW_CAND_DIR`
(scratch) rather than committed output. It gets narrowed to the official, disk-cheap
`candidate_pairs.tsv` in the next cell.


In [ ]:

# ---- Cell 11: raw retrieval for train_use + val_use -> scratch ----
def retrieve_for_entities(entity_ids, df_source, model, indexes=None):
    indexes = country_indexes if indexes is None else indexes
    rows = []
    df_e = df_source[df_source["entity_id"].isin(set(entity_ids))].copy()
    for country, group in df_e.groupby("country"):
        idx, ids, srcs = indexes.get(country, (None, np.array([]), np.array([])))
        if idx is None or len(ids) == 0:
            for eid in group["entity_id"]:
                rows.append((eid, [], []))
            continue
        q = model.encode(
            group["text"].tolist(), batch_size=ENCODE_BATCH_SIZE, normalize_embeddings=True,
            show_progress_bar=False, convert_to_numpy=True,
        ).astype("float32")
        # over-fetch then split by source so each source gets its own top-K, not a merged top-2K
        k_fetch = min(len(ids), TOP_K_PER_SOURCE * 8)
        D, I = idx.search(q, k_fetch)
        for row_i, eid in enumerate(group["entity_id"].values):
            # FAISS pads with -1 when IVF probes return fewer than k_fetch hits; ids[-1] would silently
            # wrap to the LAST corpus record, so drop padding before indexing.
            valid = I[row_i] >= 0
            hit_ids, hit_srcs, hit_scores = ids[I[row_i][valid]], srcs[I[row_i][valid]], D[row_i][valid]
            s2_hits = [(hid, sc) for hid, sr, sc in zip(hit_ids, hit_srcs, hit_scores) if sr == "S2"][:TOP_K_PER_SOURCE]
            s3_hits = [(hid, sc) for hid, sr, sc in zip(hit_ids, hit_srcs, hit_scores) if sr == "S3"][:TOP_K_PER_SOURCE]
            rows.append((eid, s2_hits, s3_hits))
    return rows

def save_raw_scores(entity_ids, out_path, df_source=None, indexes=None):
    if os.path.exists(out_path):
        print(f"{out_path} already exists, skipping.")
        return pd.read_parquet(out_path)
    df_source = s1p if df_source is None else df_source
    results = retrieve_for_entities(entity_ids, df_source, bi_model, indexes)
    score_rows = []
    for eid, s2_hits, s3_hits in results:
        for hid, sc in s2_hits + s3_hits:
            score_rows.append((eid, hid, float(sc)))
    score_df = pd.DataFrame(score_rows, columns=["source1_entity_id", "candidate_entity_id", "biencoder_score"])
    score_df.to_parquet(out_path)
    return score_df

train_raw_path = f"{RAW_CAND_DIR}/train_raw_scores.parquet"
val_raw_path = f"{RAW_CAND_DIR}/val_raw_scores.parquet"
train_raw_scores = save_raw_scores(train_ids_use, train_raw_path)
val_raw_scores = save_raw_scores(val_ids_use, val_raw_path)
print(f"Raw candidate pairs — train: {len(train_raw_scores):,}  val: {len(val_raw_scores):,}")



## 11b. Prefilter → the official `candidate_pairs.tsv`

Per the problem statement, `candidate_pairs.tsv` is defined as *"the final candidate list just before
the ML model scores them"* — not an intermediate blocking dump. So the file that actually gets saved to
`CAND_DIR` (committed output) is the bi-encoder-score top-`RERANK_PREFILTER_K` per entity: exactly what
the reranker will run on in Stage 3, and exactly what Stage 2 mines hard negatives from below — one
candidate set, used consistently for training and inference, saved once, cheaply.


In [ ]:

# ---- Cell 11b: prefilter raw scores -> official candidate_pairs.tsv (committed) ----
def prefilter_and_save(raw_scores, entity_ids, tsv_path, scores_path):
    if os.path.exists(tsv_path) and os.path.exists(scores_path):
        print(f"{tsv_path} already exists, skipping.")
        return pd.read_csv(tsv_path, sep="\t", dtype=str).fillna(""), pd.read_parquet(scores_path)
    prefiltered = (
        raw_scores.sort_values("biencoder_score", ascending=False)
        .groupby("source1_entity_id", group_keys=False)
        .head(RERANK_PREFILTER_K)
        .reset_index(drop=True)
    )
    grouped = prefiltered.groupby("source1_entity_id")["candidate_entity_id"].apply(list).to_dict()
    out_rows = [(eid, ",".join(grouped.get(eid, []))) for eid in entity_ids]
    cand_df = pd.DataFrame(out_rows, columns=["source1_entity_id", "candidate_entity_ids"])
    cand_df.to_csv(tsv_path, sep="\t", index=False)
    prefiltered.to_parquet(scores_path)
    return cand_df, prefiltered

train_cand_path = f"{CAND_DIR}/train_candidate_pairs.tsv"
train_scores_path = f"{CAND_DIR}/train_candidate_pairs_scores.parquet"
val_cand_path = f"{CAND_DIR}/val_candidate_pairs.tsv"
val_scores_path = f"{CAND_DIR}/val_candidate_pairs_scores.parquet"

train_cand_df, train_prefiltered = prefilter_and_save(train_raw_scores, train_ids_use, train_cand_path, train_scores_path)
val_cand_df, val_prefiltered_biencoder = prefilter_and_save(val_raw_scores, val_ids_use, val_cand_path, val_scores_path)
sync_to_drive(CAND_DIR)
print(train_cand_df.head(3).to_string())



## 12. Recall@k — blocking ceiling vs. what survives the prefilter

**Do this before spending any GPU time fine-tuning the reranker.** Two numbers matter here: recall
against the *raw* top-`TOP_K_PER_SOURCE` retrieval (your true blocking ceiling — nothing downstream can
recover a match blocking never found), and recall against the *prefiltered* `RERANK_PREFILTER_K` set
that actually reaches the reranker (if this is meaningfully lower than the raw number, `RERANK_PREFILTER_K`
is cutting too aggressively — raise it, at the cost of more reranker scoring later).

**Two honest caveats on what this number actually means, both worth knowing before you trust it:**

1. **FAISS IVF-PQ is an approximate search, not an exhaustive one — at full scale too, not just here.**
   Even with `DEV_MODE = False`, retrieval never literally compares an S1 entity against every single
   same-country S2/S3 record; IVF only searches the `nprobe` nearest inverted-list clusters, and PQ
   computes compressed approximate distances. That's the entire reason this recall check exists — to
   catch cases where that approximation loses a true match the exhaustive version wouldn't have.
2. **The recall number you see under `DEV_MODE = True` is an optimistic proxy, not a preview of the
   full-scale number.** The subsampled corpus (Cell 10) has far fewer distractor records per country
   than the real ~10.3M-record corpus — fewer things that can plausibly outrank a true match in the
   embedding space. Full-scale retrieval is a strictly harder search problem than DEV_MODE makes it
   look. Treat a high DEV_MODE recall as "the pipeline is wired correctly," not as "this is what
   full-scale recall will be" — you only get the real number by rerunning this cell with
   `DEV_MODE = False`.


In [ ]:

# ---- Cell 12: recall@k, raw blocking ceiling vs. post-prefilter ----
def recall_from_scores(scores_df, entity_ids, gt_map):
    hit, tot = 0, 0
    by_entity = scores_df.groupby("source1_entity_id")["candidate_entity_id"].apply(set).to_dict()
    for eid in entity_ids:
        truth = gt_map.get(eid, set())
        if len(truth) == 0:
            continue
        tot += 1
        if truth <= by_entity.get(eid, set()):
            hit += 1
    return (hit / tot if tot else float("nan")), tot

r_raw, n_raw = recall_from_scores(val_raw_scores, val_ids_use, gt_map)
r_pre, n_pre = recall_from_scores(val_prefiltered_biencoder, val_ids_use, gt_map)
print(f"Validation recall @ raw top-{TOP_K_PER_SOURCE}/source blocking:      {r_raw:.4f}  (n={n_raw:,} non-singleton entities)")
print(f"Validation recall @ prefiltered top-{RERANK_PREFILTER_K} (reranker input): {r_pre:.4f}  (n={n_pre:,})")
if r_raw < 0.95:
    print("WARNING: blocking ceiling itself is low. Consider more bi-encoder epochs or a higher TOP_K_PER_SOURCE.")
if r_raw - r_pre > 0.03:
    print(f"NOTE: the prefilter is costing you ~{(r_raw-r_pre):.1%} recall. Consider raising RERANK_PREFILTER_K.")



## 13. Stage 2 — build reranker training pairs with real hard negatives

**This is the step to get right.** Training the reranker on random pairs teaches it almost nothing —
most random (S1, S2/S3) pairs are trivially dissimilar. The candidates retrieved-but-not-matched in
Stage 1 *are* the hard negatives: near-miss legal-suffix variants, same-name-different-city records,
exactly the confusable cases the reranker will actually face at inference time. Using the same
blocking pipeline to generate both training negatives and inference candidates keeps train and
inference distributions aligned — that's what makes the reranker generalize, not what limits it.


In [ ]:

# ---- Cell 13: build labeled pairs (positives + hard negatives) for the reranker ----
rerank_train_path = f"{RERANK_DIR}/reranker_train_pairs.parquet"

if os.path.exists(rerank_train_path):
    rerank_pairs = pd.read_parquet(rerank_train_path)
else:
    rng = random.Random(SEED)
    records = []
    for _, row in train_cand_df.iterrows():
        eid = row["source1_entity_id"]
        truth = gt_map.get(eid, set())
        cand_ids = [x for x in row["candidate_entity_ids"].split(",") if x]
        positives = [c for c in cand_ids if c in truth]
        hard_negs = [c for c in cand_ids if c not in truth]
        for p in positives:
            records.append((eid, p, 1))
        # cap negatives per positive (or, for singletons, a flat cap) to keep the set balanced
        neg_cap = max(1, len(positives)) * HARD_NEGATIVES_PER_POSITIVE
        sampled_negs = rng.sample(hard_negs, min(neg_cap, len(hard_negs)))
        for nneg in sampled_negs:
            records.append((eid, nneg, 0))
    rerank_pairs = pd.DataFrame(records, columns=["source1_entity_id", "candidate_entity_id", "label"])
    rerank_pairs.to_parquet(rerank_train_path)

n_pos = (rerank_pairs["label"] == 1).sum()
n_neg = (rerank_pairs["label"] == 0).sum()
print(f"Reranker training pairs: {len(rerank_pairs):,}  (positives={n_pos:,}, negatives={n_neg:,}, ratio 1:{n_neg/max(n_pos,1):.1f})")



## 14. Stage 2 — pick a reranker: cross-encoder or GBDT

Two genuinely different, both-legitimate ways to score candidates, and it's worth choosing deliberately
rather than defaulting to the heavier one. `RERANKER_APPROACH` in the config cell controls which path
runs — both end up producing the same `reranker_score` column on the same `val_prefiltered` table, so
Cell 16 (threshold search) and the final report after it work identically either way, and every cell
below (`14a`/`15` for cross-encoder, `14b`/`15b` for GBDT) checks the flag and skips itself if it's not
selected — you can leave all of them in and just flip the one setting.

**bge-reranker-v2-m3 (cross-encoder, `Cell 14a`)** — a transformer reads the raw text of both records
together and learns the match decision end-to-end. Can pick up on subtler patterns (transliteration,
word-order transposition) than hand-written features would catch. Costs real GPU time and memory, and
you've already hit one OOM getting here.

**GBDT on engineered similarity features (`Cell 14b`)** — LightGBM trained on explicit features: token
Jaccard and edit-distance ratio for name and address separately, digit/PIN exact-overlap, length
differences, plus the bi-encoder's own cosine score as one more input. No GPU needed, trains in seconds
even on the full dataset, scores millions of candidates in minutes on CPU alone, handles the ~3.3%
missing-address records natively, and — given F_0.5 is precision-heavy — a well-calibrated threshold on
a handful of strong, literal similarity signals is often *easier* to get right than a deep model's
generalization from raw text, not a worse foundation. This class of feature-based model is a completely
standard, competitive choice for record linkage; it's not a step down.

**My actual recommendation, given what this session has already cost you:** run `Cell 14b` (GBDT) first.
Get a full pipeline working end to end with zero GPU-memory risk, see a real validation F_0.5, and treat
the cross-encoder as an upgrade to attempt afterward only if time and GPU budget allow — not the other
way around. If GBDT already validates well on your data, you may not need the transformer path at all.


In [ ]:

# ---- Cell 14b: (alternative to 14a) train a GBDT reranker on engineered similarity features ----
if RERANKER_APPROACH != "gbdt":
    print(f"RERANKER_APPROACH = {RERANKER_APPROACH!r}, skipping (this cell only runs for 'gbdt').")
else:
    import lightgbm as lgb
    import difflib

    id2name, id2addr, id2digits = {}, {}, {}
    for df in (s1p, s2p, s3p):
        id2name.update(dict(zip(df["entity_id"], df["name_norm"])))
        id2addr.update(dict(zip(df["entity_id"], df["addr_norm"])))
        id2digits.update(dict(zip(df["entity_id"], df["digits"])))

    def _jaccard(a_tokens, b_tokens):
        a, b = set(a_tokens), set(b_tokens)
        if not a and not b:
            return 1.0
        if not a or not b:
            return 0.0
        return len(a & b) / len(a | b)

    def build_gbdt_features(pairs_df, score_lookup, name_map=None, addr_map=None, digit_map=None):
        name_map = id2name if name_map is None else name_map
        addr_map = id2addr if addr_map is None else addr_map
        digit_map = id2digits if digit_map is None else digit_map
        rows = []
        for a_id, b_id in zip(pairs_df["source1_entity_id"], pairs_df["candidate_entity_id"]):
            a_name, b_name = name_map.get(a_id, ""), name_map.get(b_id, "")
            a_addr, b_addr = addr_map.get(a_id, ""), addr_map.get(b_id, "")
            a_digits = set(digit_map.get(a_id, "").split(",")) - {""}
            b_digits = set(digit_map.get(b_id, "").split(",")) - {""}
            rows.append({
                "name_jaccard": _jaccard(a_name.split(), b_name.split()),
                "addr_jaccard": _jaccard(a_addr.split(), b_addr.split()),
                "name_ratio": difflib.SequenceMatcher(None, a_name, b_name).ratio(),
                "addr_ratio": difflib.SequenceMatcher(None, a_addr, b_addr).ratio(),
                "digit_jaccard": _jaccard(a_digits, b_digits) if (a_digits or b_digits) else 0.5,
                "name_len_diff": abs(len(a_name) - len(b_name)) / max(len(a_name), len(b_name), 1),
                "addr_len_diff": abs(len(a_addr) - len(b_addr)) / max(len(a_addr), len(b_addr), 1),
                "biencoder_score": score_lookup.get((a_id, b_id), 0.0),
            })
        return pd.DataFrame(rows)

    GBDT_MODEL_PATH = f"{MODEL_DIR}/gbdt_reranker.txt"
    train_score_lookup = {
        (r.source1_entity_id, r.candidate_entity_id): r.biencoder_score
        for r in train_prefiltered.itertuples()
    }
    X_train = build_gbdt_features(rerank_pairs, train_score_lookup)
    y_train = rerank_pairs["label"]

    if os.path.exists(GBDT_MODEL_PATH):
        print("GBDT reranker already exists, loading it.")
        gbdt_booster = lgb.Booster(model_file=GBDT_MODEL_PATH)
    else:
        gbdt_model = lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=31,
            class_weight="balanced", random_state=SEED, verbose=-1,
        )
        gbdt_model.fit(X_train, y_train)
        gbdt_model.booster_.save_model(GBDT_MODEL_PATH)
        gbdt_booster = gbdt_model.booster_
        sync_to_drive(MODEL_DIR)
        print(f"GBDT reranker trained on {len(X_train):,} pairs.")
        importances = pd.Series(
            gbdt_model.feature_importances_, index=X_train.columns
        ).sort_values(ascending=False)
        print("\nFeature importances:\n", importances.to_string())


## 14a. Stage 2 — fine-tune `bge-reranker-v2-m3` as the match/no-match classifier (cross-encoder path)

In [ ]:

# ---- Cell 14a: fine-tune the cross-encoder reranker ----
if RERANKER_APPROACH != "cross_encoder":
    print(f"RERANKER_APPROACH = {RERANKER_APPROACH!r}, skipping (this cell only runs for 'cross_encoder').")
else:
    from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments
    from sentence_transformers.cross_encoder import losses as ce_losses
    import torch

    FINETUNED_RERANKER_PATH = f"{MODEL_DIR}/reranker-finetuned"

    id2text_all = id2text  # already built in Cell 8

    if os.path.exists(FINETUNED_RERANKER_PATH):
        print("Fine-tuned reranker already exists, loading it.")
        reranker = CrossEncoder(FINETUNED_RERANKER_PATH, num_labels=1, max_length=MAX_SEQ_LENGTH * 2)
    else:
        # Same reasoning as the bi-encoder's max_seq_length cap: the cross-encoder sees a CONCATENATED
        # pair (S1 text + candidate text), so its worst-case sequence is roughly double a single record's —
        # still nowhere near this model's default context window, but worth capping explicitly for the
        # same reason: bound the memory of a rare long-outlier batch, not just the typical one.
        reranker = CrossEncoder(RERANKER_BASE, num_labels=1, max_length=MAX_SEQ_LENGTH * 2)

        queries = rerank_pairs["source1_entity_id"].map(id2text_all).tolist()
        responses = rerank_pairs["candidate_entity_id"].map(id2text_all).tolist()
        labels = rerank_pairs["label"].astype(float).tolist()
        ce_train_ds = Dataset.from_dict({"query": queries, "response": responses, "label": labels})

        pos_weight = torch.tensor(n_neg / max(n_pos, 1))  # upweight the rare positive class
        loss = ce_losses.BinaryCrossEntropyLoss(reranker, pos_weight=pos_weight)

        args = CrossEncoderTrainingArguments(
            output_dir=f"{LOCAL_DIR}/reranker-ckpts",   # trainer logs/state — local scratch, not the final weights
            num_train_epochs=RERANKER_EPOCHS,
            per_device_train_batch_size=RERANKER_BATCH_SIZE,
            learning_rate=RERANKER_LR,
            warmup_ratio=0.05,
            fp16=True,
            save_strategy="no",
            logging_steps=50,
            report_to=[],
        )
        trainer = CrossEncoderTrainer(model=reranker, args=args, train_dataset=ce_train_ds, loss=loss)
        trainer.train()
        if SAVE_FP16:
            reranker.model.half()   # halves the checkpoint on disk; fine for inference-only use from here
        reranker.save_pretrained(FINETUNED_RERANKER_PATH)
        print("Saved fine-tuned reranker to", FINETUNED_RERANKER_PATH)
        sync_to_drive(MODEL_DIR)



## 15. Stage 3 — score validation candidates, tune the decision threshold

`val_candidate_pairs.tsv` (Cell 11b) is already the prefiltered, reranker-facing set — no separate
narrowing step needed here, just score every row in it with whichever reranker `RERANKER_APPROACH`
selected (Cell 15 for cross-encoder, Cell 15b for GBDT — the one that doesn't apply skips itself).


In [ ]:

# ---- Cell 15: score validation candidates with the fine-tuned cross-encoder reranker ----
if RERANKER_APPROACH != "cross_encoder":
    print(f"RERANKER_APPROACH = {RERANKER_APPROACH!r}, skipping (this cell only runs for 'cross_encoder').")
else:
    val_prefiltered = val_prefiltered_biencoder.copy()   # from Cell 11b — already at RERANK_PREFILTER_K/entity

    pair_inputs = list(zip(
        val_prefiltered["source1_entity_id"].map(id2text_all),
        val_prefiltered["candidate_entity_id"].map(id2text_all),
    ))
    rerank_scores = reranker.predict(
        pair_inputs, batch_size=256, activation_fn=torch.nn.Sigmoid(), show_progress_bar=True,
    )
    val_prefiltered["reranker_score"] = rerank_scores
    val_prefiltered.to_parquet(f"{LOCAL_DIR}/val_reranker_scores.parquet")  # local scratch — cheap to re-score if lost
    print(val_prefiltered.head(3).to_string())


In [ ]:

# ---- Cell 15b: (alternative) score validation candidates with the GBDT reranker ----
if RERANKER_APPROACH != "gbdt":
    print(f"RERANKER_APPROACH = {RERANKER_APPROACH!r}, skipping (this cell only runs for 'gbdt').")
else:
    val_prefiltered = val_prefiltered_biencoder.copy()   # from Cell 11b — already at RERANK_PREFILTER_K/entity

    val_score_lookup = {
        (r.source1_entity_id, r.candidate_entity_id): r.biencoder_score
        for r in val_prefiltered_biencoder.itertuples()
    }
    X_val = build_gbdt_features(val_prefiltered, val_score_lookup)
    val_prefiltered["reranker_score"] = gbdt_booster.predict(X_val)
    val_prefiltered.to_parquet(f"{LOCAL_DIR}/val_gbdt_scores.parquet")  # local scratch — cheap to re-score if lost
    print(val_prefiltered.head(3).to_string())


In [ ]:

# ---- Cell 16: threshold grid search to maximize macro F_0.5 on validation ----
def predictions_at_threshold(scored_df, threshold, all_val_ids):
    preds = {eid: set() for eid in all_val_ids}
    kept = scored_df[scored_df["reranker_score"] >= threshold]
    for eid, group in kept.groupby("source1_entity_id"):
        preds[eid] = set(group["candidate_entity_id"])
    return preds

best_t, best_f = None, -1.0
results = []
for t in np.round(np.arange(0.05, 0.991, 0.01), 2):   # 0.01 steps — cheap, and F_0.5 is threshold-sensitive
    preds = predictions_at_threshold(val_prefiltered, t, val_ids_use)
    rows_eval = [(gt_map.get(eid, set()), preds[eid]) for eid in val_ids_use]
    macro = macro_f_beta(rows_eval)
    results.append((t, macro))
    if macro > best_f:
        best_f, best_t = macro, t

for t, m in results:
    marker = "  <-- best" if t == best_t else ""
    print(f"threshold={t:.2f}  macro F_0.5={m:.4f}{marker}")

print(f"\nBest threshold: {best_t:.2f}  ->  validation macro F_0.5 = {best_f:.4f}")


## 16. Final validation report + save predictions in the exact submission format

In [ ]:

# ---- Cell 17: final report ----
final_preds = predictions_at_threshold(val_prefiltered, best_t, val_ids_use)

# breakdown: singleton vs non-singleton, since they behave very differently under F_0.5
singleton_scores, nonsingleton_scores = [], []
for eid in val_ids_use:
    truth = gt_map.get(eid, set())
    score = f_beta_score(truth, final_preds[eid])
    (singleton_scores if len(truth) == 0 else nonsingleton_scores).append(score)

print(f"Overall validation macro F_0.5:      {best_f:.4f}  (n={len(val_ids_use):,})")
print(f"  Singleton entities:                {np.mean(singleton_scores):.4f}  (n={len(singleton_scores):,})")
print(f"  Non-singleton entities:            {np.mean(nonsingleton_scores):.4f}  (n={len(nonsingleton_scores):,})")

# save in the exact matching_results.tsv format, for this held-out validation slice
out_rows = [(eid, ",".join(sorted(final_preds[eid]))) for eid in val_ids_use]
out_df = pd.DataFrame(out_rows, columns=["source1_entity_id", "matched_entity_ids"])
out_path = f"{LOCAL_DIR}/val_matching_results.tsv"   # small — fine to leave local, or copy to Drive yourself
out_df.to_csv(out_path, sep="\t", index=False)
print(f"\nSaved validation predictions to {out_path}")
json.dump({"best_threshold": float(best_t), "val_macro_f05": float(best_f), "run_tag": RUN_TAG,
           "reranker": RERANKER_APPROACH}, open(f"{MODEL_DIR}/threshold.json", "w"), indent=2)
sync_to_drive(MODEL_DIR)
print(f"Saved best threshold to {MODEL_DIR}/threshold.json — Section 19 uses it for the test set.")


## 19. Test-set inference → `matching_results.tsv` + `candidate_pairs.tsv`

Runs the trained pipeline on the unlabeled test set. The test S2/S3 corpus is a *different* database
from the training one, so it gets its own per-country index (`TEST_INDEX_DIR`, resumable and synced to
Drive like the training index). The bi-encoder, reranker, and threshold (`threshold.json` from Cell 17)
are reused exactly as validated — nothing is re-tuned on test data.

**Setup:** put `test_source1.tsv`, `test_source2.tsv`, `test_source3.tsv` in `TEST_INPUT_DIR` (default
`MyDrive/amazon_ml_challenge/test_data/`). Set `RUN_TEST_INFERENCE = True`. Do this after a
`DEV_MODE = False` run: DEV_MODE models are trained on a small slice and will score noticeably worse.

Entity IDs are kept in separate lookup tables from the training data on purpose, in case the test set
reuses ID strings like `S2-00047`.

Outputs are written straight to `DRIVE_DIR/submission/` (they're small).

In [ ]:
# ---- Cell 18: test-set inference -> submission files ----
RUN_TEST_INFERENCE = False
TEST_INPUT_DIR = f"{DRIVE_DIR}/test_data"
SUBMISSION_DIR = f"{DRIVE_DIR}/submission"

if not RUN_TEST_INFERENCE:
    print("RUN_TEST_INFERENCE = False, skipping. Flip it on once you're happy with the validation score.")
else:
    if DEV_MODE:
        print("WARNING: DEV_MODE=True — these models were trained on a small slice. Use a full run for submissions.")
    os.makedirs(TEST_INDEX_DIR, exist_ok=True)
    os.makedirs(SUBMISSION_DIR, exist_ok=True)
    best_t_test = json.load(open(f"{MODEL_DIR}/threshold.json"))["best_threshold"]
    print(f"Using validated threshold {best_t_test:.2f}")

    # 1. load + preprocess (cached next to the training preprocessed files)
    test_prep = {}
    for name in ("s1", "s2", "s3"):
        p = f"{PREP_DIR}/test_{name}_prep.parquet"
        if os.path.exists(p):
            test_prep[name] = pd.read_parquet(p)
        else:
            raw = pd.read_csv(f"{TEST_INPUT_DIR}/test_source{name[1]}.tsv", sep="\t", dtype=str)
            test_prep[name] = preprocess_df(raw)
            test_prep[name].to_parquet(p)
    sync_to_drive(PREP_DIR)
    t1, t2, t3 = test_prep["s1"], test_prep["s2"], test_prep["s3"]
    print(f"Test: S1={len(t1):,}  S2={len(t2):,}  S3={len(t3):,}")

    # 2. index the TEST corpus (S2+S3) per country — resumable, same builder as Cell 10
    test_corpus = pd.concat([t2.assign(source="S2"), t3.assign(source="S3")], ignore_index=True)
    test_indexes = build_or_load_indexes(test_corpus, t1["country"].unique(), TEST_INDEX_DIR, bi_model)
    sync_to_drive(TEST_INDEX_DIR)

    # 3. retrieve -> prefilter -> official candidate_pairs.tsv
    test_ids = t1["entity_id"].tolist()
    test_raw = save_raw_scores(test_ids, f"{RAW_CAND_DIR}/test_raw_scores.parquet",
                               df_source=t1, indexes=test_indexes)
    test_cand_df, test_pref = prefilter_and_save(
        test_raw, test_ids, f"{SUBMISSION_DIR}/candidate_pairs.tsv",
        f"{CAND_DIR}/test_candidate_pairs_scores.parquet",
    )

    # 4. score with the selected reranker, using TEST-only lookups
    if RERANKER_APPROACH == "gbdt":
        t_name, t_addr, t_dig = {}, {}, {}
        for df in (t1, t2, t3):
            t_name.update(dict(zip(df["entity_id"], df["name_norm"])))
            t_addr.update(dict(zip(df["entity_id"], df["addr_norm"])))
            t_dig.update(dict(zip(df["entity_id"], df["digits"])))
        t_lookup = {(r.source1_entity_id, r.candidate_entity_id): r.biencoder_score for r in test_pref.itertuples()}
        X_test = build_gbdt_features(test_pref, t_lookup, t_name, t_addr, t_dig)
        test_pref["reranker_score"] = gbdt_booster.predict(X_test) if len(X_test) else []
    else:
        t_text = {}
        for df in (t1, t2, t3):
            t_text.update(dict(zip(df["entity_id"], df["text"])))
        pair_inputs = list(zip(test_pref["source1_entity_id"].map(t_text),
                               test_pref["candidate_entity_id"].map(t_text)))
        test_pref["reranker_score"] = reranker.predict(
            pair_inputs, batch_size=256, activation_fn=torch.nn.Sigmoid(), show_progress_bar=True,
        )

    # 5. threshold -> matching_results.tsv (every test S1 entity gets a row, empty = singleton)
    test_preds = predictions_at_threshold(test_pref, best_t_test, test_ids)
    sub = pd.DataFrame([(eid, ",".join(sorted(test_preds[eid]))) for eid in test_ids],
                       columns=["source1_entity_id", "matched_entity_ids"])
    sub.to_csv(f"{SUBMISSION_DIR}/matching_results.tsv", sep="\t", index=False)

    n_empty = (sub["matched_entity_ids"] == "").sum()
    assert len(sub) == len(set(test_ids)), "duplicate S1 ids in submission"
    print(f"Wrote {len(sub):,} rows to {SUBMISSION_DIR}/matching_results.tsv "
          f"({n_empty:,} predicted singletons = {n_empty/len(sub):.1%}; train GT has ~5-6%)")
    print(sub.head().to_string())


## 17. Idle disconnects, and what to do when one happens

Colab's free tier disconnects idle sessions — leave the tab untouched too long mid-training and you
lose the runtime (a real risk during a multi-hour full-scale run, more so than Kaggle's more predictable
9-12h wall clock). There's no legitimate equivalent of Kaggle's "commit and walk away" batch mode on the
free tier; Colab Pro+ has an actual **background execution** feature for this if you need it.

This is exactly why every expensive stage in this notebook calls `sync_to_drive(...)` right after it
finishes (Cells 9, 10, 11b, 14 below) rather than only at the very end: if a disconnect happens, you
lose at most whatever the *current* stage hadn't finished, not everything since the start of the
session. When you reconnect — same notebook or a fresh one, doesn't matter — `restore_from_drive()` in
the config cell pulls all previously-synced progress back onto local disk before anything else runs, so
the resumability checks throughout the notebook pick up exactly where you left off.

Old tutorials suggesting a browser-console auto-click script to fake activity and dodge the idle timeout
are a bad idea to rely on — it's a fragile workaround for exactly the situation this sync/restore
pattern already solves properly.



## 18. Compute budget — what this actually costs, and how to scale up

Numbers below are ballpark, measured/estimated against a free-tier Colab T4 (16GB VRAM). Unlike
Kaggle's published ~30 GPU-hours/week, Colab's free-tier limits are dynamic and unpublished — expect
more variability, and heavier throttling if you've been running GPU sessions for many hours recently.

| Stage | DEV_MODE (20k S1 entities) | Full run (2.2M S1, ~10.3M S2+S3) |
|---|---|---|
| Bi-encoder fine-tune (Cell 9) | minutes | ~1-3h for 1-2 epochs over ~2M positive pairs |
| Corpus encode + IVF-PQ index build (Cell 10) | ~10-20 min | several hours — this is the single biggest cost; **it's chunked and per-country resumable, so split it across sessions if needed** |
| Raw retrieval (Cell 11) | ~1-2 min | ~1-2h for 2.2M queries against the index |
| Reranker fine-tune (Cell 14) | minutes | ~1-2h |
| Reranker scoring (Cell 15) | seconds | scoring is bounded by `RERANK_PREFILTER_K`, not `TOP_K_PER_SOURCE` — 2.2M × 50 × 2 raw candidates (~220M pairs) would not finish in a reasonable session with any transformer reranker; 2.2M × `RERANK_PREFILTER_K` (~33M pairs at K=15) is what actually makes this tractable, which is why the prefilter happens once in Cell 11b and both training (Cell 13) and scoring (Cell 15) read from that same file |

**Disk:** local scratch (`LOCAL_DIR`) has ~60GB+ free on the free tier — comfortable for everything
this notebook does locally within a session. What actually needs to survive to Drive (via
`sync_to_drive`) is just the fine-tuned models (fp16), the FAISS index, and the prefiltered candidate
files — ~5.5-6.5GB at full scale (see the itemized budget printed at the end of the config cell); check
that against your own Drive's free space (a plain free Google account gets 15GB shared with Gmail and
Photos). The FAISS IVF-PQ index alone is ~1.2GB vs. ~42GB if you tried to keep raw float32 embeddings
for the full S2+S3 corpus around, which is the biggest single reason any of this fits.

**On the "craziest model" option** — `Qwen/Qwen3-Reranker-4B` (Apache-2.0, strong multilingual
reranking) is a real, legal option under the "≤8B params" constraint and will likely edge out
`bge-reranker-v2-m3` on raw quality. Two things to know before swapping it in: (1) it's a causal-LM-based
reranker (scored via a yes/no token prompt, not a simple classification head), so fine-tuning and
inference code differ from the `CrossEncoder` class used above — treat it as a genuinely separate
integration, not a drop-in `RERANKER_BASE` string swap; (2) at ~4B params it is meaningfully slower per
pair than the 568M `bge-reranker-v2-m3` — budget accordingly, and lean harder on `RERANK_PREFILTER_K` to
keep the candidate count scoring-feasible. `Qwen3-Reranker-8B` exists too, but some sources report its
true parameter count slightly over 8B (~8.2B) — worth verifying precisely against the rules before using
it, given the stated "up to 8 billion parameters" cap and the risk of disqualification on a technicality.
